In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass
        # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# %% [markdown]
# # Step 1: Coarse Entity Retrieval (Image-to-Summary) & Wiki Image Target Extractor
# 
# This notebook implements OMGM's **Step 1** coarse retrieval pipeline on the 1,000 InfoSeek test subset:
# 1. Encodes query images using **CLIP-ViT-L/14**.
# 2. Queries the pre-built FAISS index (`summary_llama3.index`) of 100k Wikipedia summaries.
# 3. Evaluates **Recall@1, @5, @10, @20** against ground-truth Wikipedia URLs.
# 4. Saves `result.json` with top-20 retrieved candidates per query.
# 5. Filters the 13 Wiki CSVs to extract **only the specific Wikipedia images** belonging to top-20 candidates, with fixed Wikimedia URLs ready for downloading.

# %% [markdown]
# ### Cell 1: Environment Verification & Path Configuration

# %%
import os
import re
import csv
import json
import glob
import torch
import PIL.Image
from tqdm import tqdm

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
device = "cuda:0" if torch.cuda.is_available() else "cpu"


PyTorch Version: 2.11.0+cu128
CUDA Available:  True
Device Name:     Tesla T4


In [2]:
# ---------------------------------------------------------------------------
# Path Configuration (Kaggle Default Dataset Paths)
# ---------------------------------------------------------------------------
BASE_DS_DIR = "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds"

# 1. 1000 sampled test questions
SAMPLE_FILE = os.path.join(BASE_DS_DIR, "infoseek_test_sample_1000.csv")

# 2. Wikipedia 100k Knowledge Base JSON
KB_PATH = os.path.join(BASE_DS_DIR, "infoseek_100k_wiki/wiki_100_dict_v4.json")

# 3. 287MB FAISS index of Wikipedia article summaries
FAISS_INDEX_PATH = os.path.join(BASE_DS_DIR, "InfoSeek_summary_faiss_index/summary_llama3.index")


In [3]:
from huggingface_hub import login
login()

In [4]:
# 4. Extracted InfoSeek query images (983 oven_*.jpg files)
# Check both potential unzipped nesting structures
CANDIDATE_IMG_DIRS = [
    os.path.join(BASE_DS_DIR, "infoseek_val_1000_images/infoseek_val_1000_images"),
    os.path.join(BASE_DS_DIR, "infoseek_val_1000_images"),
]
QUERY_IMG_DIR = None
for p in CANDIDATE_IMG_DIRS:
    if os.path.isdir(p) and len(glob.glob(os.path.join(p, "oven_*.*"))) > 0:
        QUERY_IMG_DIR = p
        break
if QUERY_IMG_DIR is None:
    # Fallback to the first path
    QUERY_IMG_DIR = CANDIDATE_IMG_DIRS[0]

# 5. 13 Wikipedia Image URL CSVs
WIKI_CSV_DIR = os.path.join(BASE_DS_DIR, "wiki_img_output/output")

# 6. Output directory for Step 1 results
OUTPUT_DIR = "/kaggle/working/step1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n--- Verified Paths ---")
print(f"Sample File:       {SAMPLE_FILE} (Exists: {os.path.exists(SAMPLE_FILE)})")
print(f"Knowledge Base:    {KB_PATH} (Exists: {os.path.exists(KB_PATH)})")
print(f"FAISS Index:       {FAISS_INDEX_PATH} (Exists: {os.path.exists(FAISS_INDEX_PATH)})")
print(f"Query Images Dir:  {QUERY_IMG_DIR} (Exists: {os.path.isdir(QUERY_IMG_DIR)})")
print(f"Wiki CSVs Dir:     {WIKI_CSV_DIR} (Exists: {os.path.isdir(WIKI_CSV_DIR)})")
print(f"Output Directory:  {OUTPUT_DIR}")

assert os.path.exists(SAMPLE_FILE), f"Missing {SAMPLE_FILE}"
assert os.path.exists(KB_PATH), f"Missing {KB_PATH}"
assert os.path.exists(FAISS_INDEX_PATH), f"Missing {FAISS_INDEX_PATH}"




--- Verified Paths ---
Sample File:       /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_test_sample_1000.csv (Exists: True)
Knowledge Base:    /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json (Exists: True)
FAISS Index:       /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/InfoSeek_summary_faiss_index/summary_llama3.index (Exists: True)
Query Images Dir:  /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images (Exists: True)
Wiki CSVs Dir:     /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_img_output/output (Exists: True)
Output Directory:  /kaggle/working/step1_results


In [5]:
!pip install -q faiss-cpu


In [6]:
# %% [markdown]
# ### Cell 2: Core Data Structures & ClipRetriever (Matching OMGM Codebase)

# %%
import faiss
try:
    import faiss.contrib.torch_utils
except ImportError:
    pass
from transformers import AutoModel, AutoProcessor


import faiss
from transformers import AutoModel, CLIPImageProcessor


class WikipediaKnowledgeBaseEntry:
    """Represents a single Wikipedia article entry in the knowledge base."""
    def __init__(self, entry_dict):
        self.title = entry_dict.get("title", "")
        self.url = entry_dict.get("url", "")
        self.image_urls = entry_dict.get("image_urls", [])
        self.image_reference_descriptions = entry_dict.get("image_reference_descriptions", [])
        self.image_section_indices = entry_dict.get("image_section_indices", [])
        self.section_titles = entry_dict.get("section_titles", [])
        self.section_texts = entry_dict.get("section_texts", [])
        self.image = {}
        self.score = {}


class ClipRetriever:
    """Step 1 Image-to-Summary Retriever with EVA-CLIP (1280-dim) on cuda:0."""
    def __init__(self, model_type="eva-clip", device="cuda:0"):
        self.device = device
        self.model_type = model_type
        print(f"Loading retriever model: {model_type} onto {device}...")
        
        # Dedicated image processor
        self.processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
        
        if model_type == "eva-clip":
            # 1. Load onto CPU first so we can strip the text model before touching GPU
            self.model = AutoModel.from_pretrained(
                "BAAI/EVA-CLIP-8B",
                torch_dtype=torch.float16,
                trust_remote_code=True,
            )
            # 2. Delete text components to reduce VRAM down to ~8.8GB (vision only)
            if hasattr(self.model, "text_projection"):
                del self.model.text_projection
            if hasattr(self.model, "text_model"):
                del self.model.text_model
            
            # 3. Move strictly to cuda:0 so all vision layers are on the same device
            self.model.to(self.device).eval()

            # 4. FIX: BAAI persistent=False bug - explicitly set position_ids on cuda:0
            if hasattr(self.model, "vision_model") and hasattr(self.model.vision_model, "embeddings"):
                num_pos = self.model.vision_model.embeddings.num_positions
                self.model.vision_model.embeddings.position_ids = torch.arange(num_pos, device=self.device).unsqueeze(0)
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            self.model = AutoModel.from_pretrained(
                "openai/clip-vit-large-patch14",
                torch_dtype=torch.float16,
            )
            if hasattr(self.model, "text_projection"):
                del self.model.text_projection
            if hasattr(self.model, "text_model"):
                del self.model.text_model
            self.model.to(self.device).eval()
            
        print(f"Model successfully loaded on: {self.device}")
        self.knowledge_base = []
        self.entity_faiss_index = None

    def load_knowledge_base(self, kb_path):
        print(f"Loading Knowledge Base from {kb_path}...")
        with open(kb_path, "r", encoding="utf-8") as f:
            kb_dict = json.load(f)
        
        self.knowledge_base = [WikipediaKnowledgeBaseEntry(entry) for entry in kb_dict.values()]
        print(f"Loaded Knowledge Base with {len(self.knowledge_base):,} entries.")

    def load_entity_faiss_index(self, index_path):
        print(f"Loading FAISS index from {index_path}...")
        self.entity_faiss_index = faiss.read_index(index_path)
        print(f"FAISS index ready on CPU with {self.entity_faiss_index.ntotal:,} vectors (dim={self.entity_faiss_index.d}).")

    @torch.no_grad()
    def I2T_faiss(self, image: PIL.Image.Image, top_k: int = 20):
        """Retrieve top-K most similar Wikipedia article summaries using query image."""
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs.pixel_values.to(self.device).half()
        
        if self.model_type == "eva-clip":
            image_score = self.model.encode_image(pixel_values)
        else:
            image_score = self.model.get_image_features(pixel_values=pixel_values)
            if hasattr(image_score, "pooler_output") and image_score.pooler_output is not None:
                image_score = image_score.pooler_output
            elif hasattr(image_score, "image_embeds") and image_score.image_embeds is not None:
                image_score = image_score.image_embeds
            elif not isinstance(image_score, torch.Tensor):
                image_score = image_score[0]

        query = image_score.float()
        query = torch.nn.functional.normalize(query, dim=-1)

        # Convert to numpy for CPU FAISS search (zero GPU VRAM usage)
        query_np = query.detach().cpu().numpy().astype("float32")
        D, I = self.entity_faiss_index.search(query_np, top_k)

        top_k_entries = []
        for i in range(top_k):
            idx = int(I[0][i])
            kb_entry = self.knowledge_base[idx]
            sim = float(D[0][i])
            top_k_entries.append({
                "url": kb_entry.url,
                "title": kb_entry.title,
                "knowledge_base_index": idx,
                "similarity": sim,
                "kb_entry": kb_entry,
            })
        return top_k_entries


In [7]:
def load_csv_data(sample_file):
    sample_list = []
    with open(sample_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            sample_list.append(row)
    return sample_list, header


def get_image_path(img_id, search_dir):
    """Finds image with .jpg, .JPEG, .png or direct filename."""
    img_id = img_id.split("|")[0].strip()
    candidates = [
        os.path.join(search_dir, f"{img_id}.jpg"),
        os.path.join(search_dir, f"{img_id}.JPEG"),
        os.path.join(search_dir, f"{img_id}.png"),
        os.path.join(search_dir, img_id),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Query image {img_id} not found in {search_dir}")


def remove_list_duplicates(lst):
    seen = set()
    return [x for x in lst if not (x in seen or seen.add(x))]


def eval_recall(candidates, ground_truth, top_ks=[1, 5, 10, 20]):
    recall = {k: 0 for k in top_ks}
    for k in top_ks:
        if ground_truth in candidates[:k]:
            recall[k] = 1
    return recall



In [8]:

# %% [markdown]
# ### Cell 3: Run Step 1 Coarse Retrieval & Compute Recall@K

# %%
RETRIEVAL_TOP_K = 20
TOP_KS = [1, 5, 10, 20]

# 1. Initialize Retriever
retriever = ClipRetriever(model_type="eva-clip", device=device)
retriever.load_knowledge_base(KB_PATH)
retriever.load_entity_faiss_index(FAISS_INDEX_PATH)




Loading retriever model: eva-clip onto cuda:0...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/875 [00:00<?, ?it/s]

Model successfully loaded on: cuda:0
Loading Knowledge Base from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json...
Loaded Knowledge Base with 100,000 entries.
Loading FAISS index from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/InfoSeek_summary_faiss_index/summary_llama3.index...
FAISS index ready on CPU with 100,000 vectors (dim=1280).


In [9]:
# 2. Load Sample Test Queries
sample_list, sample_header = load_csv_data(SAMPLE_FILE)
print(f"\nLoaded {len(sample_list):,} sample test questions.")

recalls = {k: 0 for k in TOP_KS}
retrieval_results = {}

print(f"\nRunning Step 1 Img2Sum Retrieval (top_k={RETRIEVAL_TOP_K})...")
for it, row in tqdm(enumerate(sample_list), total=len(sample_list), desc="Step 1 Retrieval"):
    example = {sample_header[i]: row[i] for i in range(len(sample_header))}
    ground_truth_url = example["wikipedia_url"].strip()
    data_id = example["data_id"]
    raw_img_id = example["dataset_image_ids"]

    # Load query image
    img_path = get_image_path(raw_img_id, QUERY_IMG_DIR)
    image = PIL.Image.open(img_path).convert("RGB")

    # Run I2T FAISS search
    top_k = retriever.I2T_faiss(image, top_k=RETRIEVAL_TOP_K)
    
    top_k_urls = [entry["url"] for entry in top_k]
    top_k_urls = remove_list_duplicates(top_k_urls)

    # Save retrieval result format identical to OMGM
    seen_urls = set()
    dedup_entries = []
    dedup_sims = []
    for item in top_k:
        if item["url"] not in seen_urls:
            seen_urls.add(item["url"])
            dedup_entries.append({"url": item["url"], "title": item["title"]})
            dedup_sims.append(item["similarity"])

    retrieval_results[data_id] = {
        "question": example.get("question", ""),
        "ground_truth_url": ground_truth_url,
        "retrieved_entities": dedup_entries[:RETRIEVAL_TOP_K],
        "retrieval_similarities": dedup_sims[:RETRIEVAL_TOP_K],
    }

    # Evaluate recall
    recall = eval_recall(top_k_urls, ground_truth_url, TOP_KS)
    for k in TOP_KS:
        recalls[k] += recall[k]

total_queries = len(sample_list)
print("\n" + "="*45)
print(f"STEP 1 RETRIEVAL EVALUATION RESULTS (N={total_queries})")
print("="*45)
for k in TOP_KS:
    avg_rec = (recalls[k] / total_queries) * 100.0
    print(f"Avg Recall@{k:2d}: {avg_rec:6.2f}% ({recalls[k]}/{total_queries})")
print("="*45)

# Save result.json
result_json_path = os.path.join(OUTPUT_DIR, "result.json")
with open(result_json_path, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=2)
print(f"\nSaved Step 1 retrieval results to: {result_json_path}")



Loaded 1,000 sample test questions.

Running Step 1 Img2Sum Retrieval (top_k=20)...


Step 1 Retrieval: 100%|██████████| 1000/1000 [06:06<00:00,  2.73it/s]


STEP 1 RETRIEVAL EVALUATION RESULTS (N=1000)
Avg Recall@ 1:  49.50% (495/1000)
Avg Recall@ 5:  71.50% (715/1000)
Avg Recall@10:  78.80% (788/1000)
Avg Recall@20:  83.40% (834/1000)

Saved Step 1 retrieval results to: /kaggle/working/step1_results/result.json


In [10]:
print("FAISS index dimension (self.d):", retriever.entity_faiss_index.d)

FAISS index dimension (self.d): 1280


In [11]:
def fix_wikimedia_url(url: str) -> str:
    """
    Transforms deprecated Wikimedia thumbnail URLs to original image URLs.
    Stripping /thumb/ and the trailing size suffix avoids HTTP 400 Bad Request.
    """
    pattern = r"(https?://upload\.wikimedia\.org/wikipedia/[^/]+)/thumb/(.+)/[^/]+$"
    match = re.match(pattern, url)
    if match:
        return f"{match.group(1)}/{match.group(2)}"
    return url


# 1. Collect all unique Wikipedia titles and URLs appearing in Top-20
unique_candidate_titles = set()
unique_candidate_urls = set()

for q_id, q_data in retrieval_results.items():
    for ent in q_data["retrieved_entities"]:
        if ent.get("title"):
            unique_candidate_titles.add(ent["title"].strip())
        if ent.get("url"):
            unique_candidate_urls.add(ent["url"].strip())

print("\n--- Candidate Entities Summary ---")
print(f"Total Queries Evaluated:            {len(retrieval_results):,}")
print(f"Total Candidate Mentions:           {len(retrieval_results) * RETRIEVAL_TOP_K:,}")
print(f"Unique Wikipedia Entities in Top-20: {len(unique_candidate_titles):,}")

# 2. Scan the 13 Wiki CSVs and filter rows for our unique candidates
wiki_csv_files = sorted(glob.glob(os.path.join(WIKI_CSV_DIR, "wiki_image_url_part_*_processed.csv")))
print(f"\nFound {len(wiki_csv_files)} Wikipedia image metadata CSV files.")

matched_images = []
seen_img_paths = set()

for csv_path in tqdm(wiki_csv_files, desc="Filtering Wiki Image CSVs"):
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            if not row or len(row) < 4:
                continue
            title = row[0].strip()
            raw_url = row[1].strip()
            relative_path = row[2].strip()
            caption = row[3].strip()

            if title in unique_candidate_titles:
                if relative_path not in seen_img_paths:
                    seen_img_paths.add(relative_path)
                    fixed_url = fix_wikimedia_url(raw_url)
                    matched_images.append({
                        "title": title,
                        "original_url": raw_url,
                        "download_url": fixed_url,
                        "relative_path": relative_path,
                        "caption": caption
                    })

print("\n" + "="*45)
print("FILTERED WIKI IMAGES SUMMARY")
print("="*45)
print(f"Unique Top-20 Entities:             {len(unique_candidate_titles):,}")
print(f"Total Target Wiki Images to Download: {len(matched_images):,}")
avg_imgs_per_entity = len(matched_images) / max(len(unique_candidate_titles), 1)
print(f"Average Images per Entity:          {avg_imgs_per_entity:.2f}")
# Assuming average image size ~150 KB
estimated_size_mb = (len(matched_images) * 150) / 1024
print(f"Estimated Download Size:            ~{estimated_size_mb:.1f} MB ({estimated_size_mb/1024:.2f} GB)")
print("="*45)

# Save filtered target images CSV
target_images_csv = os.path.join(OUTPUT_DIR, "candidate_wiki_images_to_download.csv")
with open(target_images_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "download_url", "relative_path", "caption", "original_url"])
    writer.writeheader()
    writer.writerows(matched_images)

print(f"\nSaved download manifest to: {target_images_csv}")

# Preview first 3 entries
print("\nPreview of first 3 target image download entries:")
for i, item in enumerate(matched_images[:3], 1):
    print(f"[{i}] Title: {item['title']}")
    print(f"    URL:   {item['download_url']}")
    print(f"    Path:  {item['relative_path']}")
    print(f"    Desc:  {item['caption'][:60]}...\n")



--- Candidate Entities Summary ---
Total Queries Evaluated:            1,000
Total Candidate Mentions:           20,000
Unique Wikipedia Entities in Top-20: 11,657

Found 13 Wikipedia image metadata CSV files.


Filtering Wiki Image CSVs: 100%|██████████| 13/13 [00:32<00:00,  2.54s/it]



FILTERED WIKI IMAGES SUMMARY
Unique Top-20 Entities:             11,657
Total Target Wiki Images to Download: 42,907
Average Images per Entity:          3.68
Estimated Download Size:            ~6285.2 MB (6.14 GB)

Saved download manifest to: /kaggle/working/step1_results/candidate_wiki_images_to_download.csv

Preview of first 3 target image download entries:
[1] Title: Thick-billed kingbird
    URL:   https://upload.wikimedia.org/wikipedia/commons/1/11/Thick-billed_Kingbird_%286062989988%29.jpg
    Path:  full/wiki_image_split/wiki_entity_image_10/Thick-billed kingbird/220px-Thick-billed_Kingbird_%286062989988%29.jpg
    Desc:  Thick-billed Kingbird (6062989988)...

[2] Title: Thick-billed kingbird
    URL:   https://upload.wikimedia.org/wikipedia/commons/5/5a/Status_iucn3.1_LC.svg
    Path:  full/wiki_image_split/wiki_entity_image_10/Thick-billed kingbird/220px-Status_iucn3.1_LC.svg.png
    Desc:  Status iucn3.1 LC...

[3] Title: Thick-billed kingbird
    URL:   https://upload.wiki